## 1. Activation Functions

### SwiGLU
**Formula**  
${
\text{SwiGLU}(x) = (x \cdot \sigma(\beta x)) \otimes (x W_1 + b_1) \quad \text{(often $\beta=1$)}
}$

Typically applied as `Swish(xW_gate) * (xW_up)`, where `Swish(x) = x·σ(x)`.

**When to use**  
- Transformer FFN layers (e.g., LLaMA, PaLM). It improves training stability and downstream performance over ReLU/GELU.
- When you can afford ~3/2× the parameters of a standard FFN (it needs a separate gate projection).

**When NOT to use**  
- Tight memory/compute budgets (doubles the linear projection cost).  
- Very shallow networks where the benefit is marginal.

**PyTorch implementation**  
```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, bias=False):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=bias)   # up
        self.w2 = nn.Linear(d_model, d_ff, bias=bias)   # gate
        self.w3 = nn.Linear(d_ff, d_model, bias=bias)   # down
    def forward(self, x):
        return self.w3(F.silu(self.w2(x)) * self.w1(x))   # SiLU = Swish
```

---

### Leaky-ReLU
**Formula**  
${
\text{LeakyReLU}(x) = \max(0, x) + \alpha \cdot \min(0, x), \quad \alpha \approx 0.01
}$

**When to use**  
- Preventing “dying ReLU” in deep CNNs or dense networks.  
- When you want a negligible extra cost over ReLU.

**When NOT to use**  
- When more advanced activations (GELU, Swish) give better convergence (common in Transformers).  
- If model performance is already saturated with ReLU.

**PyTorch**  
```python
nn.LeakyReLU(negative_slope=0.01)   # built-in
```

---

### Sigmoid
**Formula**  
${
\sigma(x) = \frac{1}{1 + e^{-x}}
}$

**When to use**  
- Binary classification output.  
- Gating mechanisms (LSTM gates, attention gates, SwiGLU’s gating).  
- Converting logits to probabilities for single‑label tasks.

**When NOT to use**  
- Hidden layers of deep networks → vanishing gradients saturate.  
- Multi‑class output (use softmax instead).

**PyTorch**  
```python
torch.sigmoid(x)
```

---

### Tanh
**Formula**  
${
\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}
}$

**When to use**  
- RNN/LSTM cell state updates (standard).  
- Outputting values in [-1, 1] (e.g., normalising flows, certain RL policies).  
- Occasionally in attention (pre‑softmax).

**When NOT to use**  
- Deep feed‑forward hidden layers → gradient saturation same as sigmoid.  
- When output must be positive or in (0,1).

**PyTorch**  
```python
torch.tanh(x)
```

---


## 2. Normalisation

### RMSNorm
**Formula**  
${
\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^d x_i^2 + \epsilon}} \odot \gamma
}$

No re‑centering (no mean subtraction), only scaling.

**When to use**  
- LLM Transformers (LLaMA, Mistral) – computationally cheaper, equally effective as LayerNorm.
- When you want to reduce training variance with fewer parameters (no β bias).

**When NOT to use**  
- When mean centering is crucial (e.g., some vision tasks).  
- Extremely small batch/dimension where the root‑mean‑square estimate becomes unstable (rare).

**PyTorch implementation**  
```python
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight
```

---

### LayerNorm
**Formula**  
${
\text{LayerNorm}(x) = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \odot \gamma + \beta
}$

where μ, σ² are computed over the last D dimensions.

**When to use**  
- Transformers (standard pre‑LN).  
- RNNs, and whenever batch size is small/variable.

**When NOT to use**  
- Large‑batch CNN training where BatchNorm gives a stronger regularising effect.  
- When you need per‑feature normalisation across the batch (BatchNorm).

**PyTorch**  
```python
nn.LayerNorm(normalized_shape)
```

---

### BatchNorm
**Formula**  
${
\text{BatchNorm}(x) = \frac{x - \mu_B}{\sqrt{\sigma^2_B + \epsilon}} \odot \gamma + \beta
}$

μ_B, σ²_B computed over the batch and spatial dimensions.

**When to use**  
- CNNs (VGG, ResNet) with sufficiently large batch size (≥16).  
- When you want to reduce internal covariate shift and benefit from the noise of batch statistics (regularisation).

**When NOT to use**  
- Transformers (unstable with variable sequence length, small batch).  
- When batch size is 1 or very small.  
- Recurrent models (different statistics per time step).

**PyTorch**  
```python
nn.BatchNorm1d(num_features)  # 1D
nn.BatchNorm2d(num_features)  # 2D (images)
```


## 3. Optimisers

### AdamW
**Update rule** (decoupled weight decay)  
${
\begin{aligned}
m_t &= \beta_1 m_{t-1} + (1-\beta_1) g_t \\
v_t &= \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \\
\hat{m}_t &= m_t/(1-\beta_1^t) \\
\hat{v}_t &= v_t/(1-\beta_2^t) \\
\theta_{t} &= \theta_{t-1} - \eta \left( \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} + \lambda \theta_{t-1} \right)
\end{aligned}
}$
**When to use**  
- Default for Transformers, LLMs, and most modern deep learning.  
- Handles sparse gradients, scales well, robust to hyperparameters.

**When NOT to use**  
- When you need extremely fine‑grained control over learning dynamics (some RL, fine‑tuning with tiny data where SGD + momentum might generalise slightly better).  
- Memory‑constrained edge devices that cannot afford second‑moment estimates (though Adam can be made efficient).

**PyTorch**  
```python
torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
```

---

### SGD (with momentum)
**Update rule**  
${
\begin{aligned}
v_t &= \mu v_{t-1} + g_t \\
\theta_t &= \theta_{t-1} - \eta v_t
\end{aligned}
}$

(plus optional weight decay, usually L2‑coupled).

**When to use**  
- Classic computer vision (ResNet training recipe).  
- When you prefer well‑understood generalisation properties (large batch scenarios).

**When NOT to use**  
- Transformers/LLMs – they typically require AdamW for stable, fast convergence.  
- When gradients are sparse or noisy (Adam’s adaptive rates help).

**PyTorch**  
```python
torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
```

---



## 4. Loss Functions

### Cross‑Entropy (CE)
**Formula** (multi‑class)  
${
\mathcal{L}_{CE} = -\sum_{c=1}^C y_c \log(\hat{y}_c)
}$

where ${(\hat{y} = \text{softmax}(z))}$.

**When to use**  
- Multi‑class classification (language modelling, image classification).  
- When labels are one‑hot (or probability distributions for knowledge distillation).

**When NOT to use**  
- Regression tasks (use MSE).  
- Binary classification when you can use BCEWithLogitsLoss (more stable).  
- When you need to model ordinal relationships (e.g., age estimation).

**PyTorch**  
```python
nn.CrossEntropyLoss()                         # logits (N,C) + class indices
# For language modelling with ignore_index
nn.CrossEntropyLoss(ignore_index=pad_token_id)
```

---

### Mean Squared Error (MSE)
**Formula**  
${\
\mathcal{L}_{MSE} = \frac{1}{N}\sum_{i=1}^N (y_i - \hat{y}_i)^2
}$

**When to use**  
- Regression (predicting continuous values).  
- As a reconstruction loss in autoencoders.  
- When outliers are not a dominant problem (sensitive to large errors).

**When NOT to use**  
- Classification (bad probability calibration).  
- When you want robustness to outliers (use Huber loss / MAE).  
- Outputs bounded in [0,1] (may use BCE instead).

**PyTorch**  
```python
nn.MSELoss()
```

---



## 5. Positional Encodings

### RoPE (Rotary Position Embedding)
**Formula**  
For query/key vectors at positions m, n:
${
\tilde{q}_m = R^d_{\Theta, m} q_m, \quad \tilde{k}_n = R^d_{\Theta, n} k_n
}$

with block‑diagonal rotation matrices encoding relative position m–n in dot product.  

Theta frequencies: ${(\theta_i = b^{-2i/d}, b=10000)}$.

**When to use**  
- LLMs (LLaMA, GPT‑NeoX, Mistral) – enables long‑range extrapolation and relative position encoding with no extra parameters.  
- When you want the model to generalise beyond training sequence length (with proper scaling).

**When NOT to use**  
- Non‑transformer architectures where absolute position matters more.  
- When you need strictly absolute position information (though RoPE carries both absolute and relative).

**PyTorch implementation** (simplified):
```python
def precompute_freqs_cis(dim, end, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(end)
    freqs = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(freqs), freqs)  # complex rotation

def apply_rotary_emb(xq, xk, freqs_cis):
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = freqs_cis.unsqueeze(0).unsqueeze(0)  # [1, 1, T, dim//2]
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)
```

---

### ALiBi (Attention with Linear Biases)
**Formula**  
Add a static, non‑learned bias to the attention scores:
${
\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + m \cdot B\right) V
}$

where ${(B_{ij} = -|i-j|)}$ (or multiple slopes per head).

**When to use**  
- Efficient length extrapolation without extra parameters.  
- When you want a simple, head‑specific relative position cue.

**When NOT to use**  
- When RoPE already delivers better performance (most recent LLMs prefer RoPE).  
- For tasks that need learnable relative biases (T5‑style).

**PyTorch snippet** (adding bias inside attention):
```python
def get_alibi_slopes(n_heads):
    # geometric sequence: 2^{-8/n}, 2^{-16/n}, ...
    return torch.tensor([2**(-8 * i / n_heads) for i in range(1, n_heads+1)])

def apply_alibi(scores, alibi_slopes, seq_len):
    bias = torch.arange(seq_len).unsqueeze(0) - torch.arange(seq_len).unsqueeze(1)
    bias = -torch.abs(bias).unsqueeze(0)  # [1, T, T]
    bias = bias * alibi_slopes.view(-1, 1, 1)
    return scores + bias
```

---

### Sinusoidal (original Transformer)
**Formula**  
${
\begin{aligned}
PE_{(pos, 2i)} &= \sin(pos / 10000^{2i/d}) \\
PE_{(pos, 2i+1)} &= \cos(pos / 10000^{2i/d})
\end{aligned}
}$

**When to use**  
- Absolute position encoding baseline.  
- When training data length is fixed and you don’t need extrapolation.  
- Simple sequence‑to‑sequence models.

**When NOT to use**  
- When you need long‑range generalisation (RoPE/ALiBi are better).  
- Modern LLMs – almost completely replaced by RoPE.

**PyTorch**  
```python
class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]
```

---



## 6. Mixture‑of‑Experts (MoE) Feed‑Forward Network

**Formula** (simplified, top‑k routing)  

${
\text{MoE-FFN}(x) = \sum_{i \in \mathcal{T}} \text{softmax}(x W_g)_i \cdot \text{FFN}_i(x)
}$

where ${(\mathcal{T})}$ is the set of top‑k experts chosen by the gating scores, and ${(W_g \in \mathbb{R}^{d \times E})}$.

**When to use**  
- Scaling model capacity without proportionally increasing compute (Mixtral, GPT‑4 style).  
- When you have the infrastructure for distributed expert parallelism.

**When NOT to use**  
- Latency‑sensitive applications (all‑to‑all communication overhead).  
- Small models where the overhead of routing negates benefits.  
- When training data is insufficient to specialise many experts.

**PyTorch implementation** (top‑2 gating, with load balancing loss):
```python
class MoE_FFN(nn.Module):
    def __init__(self, d_model, d_ff, n_experts, top_k=2, dropout=0.1):
        super().__init__()
        self.top_k = top_k
        self.n_experts = n_experts
        self.gate = nn.Linear(d_model, n_experts, bias=False)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d_ff, d_model)
            ) for _ in range(n_experts)
        ])

    def forward(self, x):
        # x: [B, T, D]
        B, T, D = x.shape
        gate_logits = self.gate(x)  # [B, T, E]
        weights, idx = torch.topk(gate_logits, self.top_k, dim=-1)  # [B, T, K]
        weights = F.softmax(weights, dim=-1)

        out = torch.zeros_like(x)
        for i, expert in enumerate(self.experts):
            # mask positions where expert i is in top-k
            mask = (idx == i).any(dim=-1)   # [B, T]
            if mask.any():
                expert_input = x[mask]      # [N, D]
                expert_out = expert(expert_input)  # [N, D]
                # select the weight corresponding to this expert
                weight_idx = (idx == i).float()  # [B, T, K] (one-hot over K)
                selected_weights = (weight_idx * weights.unsqueeze(-1)).sum(dim=-1)  # [B, T]
                out[mask] += selected_weights[mask].unsqueeze(-1) * expert_out
        return out
```

**Note:** In production MoE, you would implement an efficient batched dispatch/combine to avoid the Python loop over experts, but the above illustrates the principle.

---



Here's a concise table mapping each building block to its role in the bias–variance tradeoff, based on the detailed discussion above:

| Building Block            | Effect on Bias                                                                 | Effect on Variance                                                                 | How It Manages the Tradeoff                                                                                       |
|---------------------------|--------------------------------------------------------------------------------|------------------------------------------------------------------------------------|------------------------------------------------------------------------------------------------------------------|
| **Activations**           | Sigmoid/tanh can increase bias (saturation); Leaky‑ReLU slightly reduces bias by preventing dead neurons; SwiGLU/GELU keep bias low via smooth curvature. | Smooth activations (SwiGLU, GELU) reduce gradient variance; Leaky‑ReLU avoids dying‑neuron variance spikes. | Smoother functions (SwiGLU, GELU) lower variance and allow stable training of low‑bias large models.              |
| **Normalisation**         | Enables training deeper/larger models → indirect **bias reduction** (higher capacity). | Directly **reduces variance** by stabilising feature distributions and providing slight regularisation noise (BatchNorm). | RMSNorm/LayerNorm: low‑cost variance suppression → can safely lower bias by scaling model size.                   |
| **Optimiser**             | AdamW fits complex patterns well → **low bias**; SGD may converge to simpler solutions (slightly higher bias) but often generalises better on small data. | AdamW reduces variance of gradient steps via momentum & adaptive rates; SGD has higher update variance but can find flatter minima. | AdamW is the default for overparametrised models (variance control + bias reduction); SGD used when variance‑prone but sharp minima avoidance matters. |
| **Loss function**         | MSE assumes constant noise → high bias if noise is heteroscedastic. CE is well‑calibrated for classification → low bias. | MSE is sensitive to outliers (high variance). CE with softmax has controlled output variance. | Choose CE for classification (balanced bias‑variance); MSE for Gaussian‑noise regression; use robust losses (Huber) to reduce variance from outliers. |
| **Positional encoding**   | RoPE/ALiBi reduce **bias from finite context** by enabling length extrapolation; sinusoidal PE has limited range (higher bias on long sequences). | RoPE/ALiBi have no trainable parameters → **no added variance**. Sinusoidal PE likewise zero variance. | RoPE/ALiBi lower bias without inflating variance, outperforming learnable or fixed absolute encodings in LLMs.    |
| **MoE‑FFN**               | Massive capacity increase with many experts → **sharp reduction in bias** (model can learn more diverse patterns). | Sparse routing (only top‑k experts active per token) **keeps variance in check** by avoiding overfitting; load‑balancing loss further regulates expert usage. | Tradeoff: sparsity controls variance while expert count lowers bias. Ideal for scaling without linear compute growth. |

**When to apply each choice** follows directly:  
- For LLMs, use **SwiGLU + RMSNorm + AdamW + CE + RoPE + (optional) MoE** to maximise bias reduction while tightly controlling variance.  
- For small‑data CNNs, **BatchNorm + SGD + MSE/CE** and simpler activations (Leaky‑ReLU) often suffice, because the extreme overparametrised regime is not reached.